In [3]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

GPU count: 3
0 NVIDIA RTX A6000
1 NVIDIA RTX A6000
2 NVIDIA RTX A6000


In [4]:
from configs.experiment_config import (
    MODEL_NAME,
    SEED,
    MAX_NEW_TOKENS,
    STEPS,
    TEMPERATURE,
    TOP_P,
    TOP_K,
    ALGORITHM,
    PROMPT,
    SPLIT_LAYERS,
    WARMUP,
    REPEATS,
    EDGE_DEVICE,
    CLOUD_DEVICE,
)

print("EDGE_DEVICE =", EDGE_DEVICE)
print("CLOUD_DEVICE =", CLOUD_DEVICE)

EDGE_DEVICE = cuda:2
CLOUD_DEVICE = cuda:0


In [10]:
from src.model_loader import load_dream_model
from src.generation import generate_with_token_history

from configs.experiment_config import (
    MODEL_NAME,
    PROMPT,
    MAX_NEW_TOKENS,
    STEPS,
    TEMPERATURE,
    TOP_P,
    TOP_K,
    ALGORITHM,
)

# Load model + tokenizer
model, tokenizer = load_dream_model(MODEL_NAME)
model.eval()


def create_token_history_debug_hook():
    history = []
    steps_seen = []

    def hook(step, x, logits):
        history.append(
            x[0].detach().clone().cpu()
        )

        steps_seen.append(step)

        return x

    return hook, history, steps_seen


hook, history, steps_seen = (
    create_token_history_debug_hook()
)

_ = generate_with_token_history(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=TEMPERATURE,
    history_hook=hook,
    top_p=TOP_P,
    top_k=TOP_K,
    alg=ALGORITHM,
)

print("Captured states:", len(history))
print("Number of hook calls:", len(steps_seen))
print("Steps seen:", steps_seen)

if len(steps_seen) > 0:
    print("First step:", steps_seen[0])
    print("Last step:", steps_seen[-1])

/mnt/mydisk/mery-splitdllm/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  6.19it/s]
/mnt/mydisk/mery-splitdllm/env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/mnt/mydisk/mery-splitdllm/env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperat

Captured states: 65
Number of hook calls: 65
Steps seen: [None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]
First step: None
Last step: 63


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "GSAI-ML/LLaDA-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).cuda().eval()

print(type(model))
print("Loaded:", MODEL_NAME)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.